In [ ]:
# Install dependencies
!pip install anthropic python-dotenv

In [ ]:
# Load env
from dotenv import load_dotenv

load_dotenv()

In [6]:
# API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [7]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0]


In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

In [ ]:
# Chatbot
messages = []

while True:
    user_input = input("You: ")
    print(f"You: {user_input}")

    add_user_message(messages, user_input)
    response = chat(messages)
    print(f"Claude: {response}")
    
    add_assistant_message(messages, response)

In [ ]:
messages = []

system ="""
    You are a patient math tutor.
    Do not directly answer the questions.
    Guide to a solution step by step
    """

add_user_message(messages, "What is 2+2?")
answer = chat(messages, system=system)
print(answer)

In [ ]:
messages = []

add_user_message(messages, "Generate a sentence movie idea")
answer = chat(messages, temperature=0.0)
print(answer)

In [ ]:
message = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

answer = chat(messages, stop_sequences="```")
print(answer)

# Prompt Evaluation

In [ ]:
import json

def generate_dataset():
    prompt = """
            Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

            Example output:
            ```json
            [
            {
                "task": "Description of task",
                "format": "python/json/regex",
                "solution_criteria": "Criteria for evaluating the solution"
            },
            ...additional
            ]
            ```

            * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
            * Focus on tasks that do not require writing much code

            Please generate 3 objects.
            """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    response = chat(messages, stop_sequences="```")

    return json.loads(response)

In [ ]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}

    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    response = chat(messages, stop_sequences="```")
    return response

In [ ]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    Criteria: {criteria}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [ ]:
import json
import ast
import re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(output, test_case):
    format_type = test_case["format"]
    if format_type == "json":
        return validate_json(output)
    elif format_type == "python":
        return validate_python(output)
    elif format_type == "regex":
        return validate_regex(output)
    else:
        return 0

In [ ]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [ ]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average Score: {average_score}")
    
    return results

In [ ]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))